# Notebook 21 — Preference Data and Direct Preference Optimization

    ## Learning objectives

    - Represent chosen/rejected preference pairs and identify data pathologies
- Explain the DPO objective relative to a reference policy
- Configure a guarded TRL DPO experiment and evaluate behavior

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

# VS Code's Colab extension can attach to a Colab kernel before `google.colab`
# has been imported, so checking only sys.modules produces a false negative.
try:
    HAS_GOOGLE_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # The parent `google` namespace is absent locally.
    HAS_GOOGLE_COLAB = False
IN_COLAB = HAS_GOOGLE_COLAB or bool(os.getenv("COLAB_RELEASE_TAG")) or bool(os.getenv("COLAB_GPU"))
PACKAGES = ['transformers>=4.51,<5', 'datasets>=3.5,<6', 'peft>=0.15', 'trl>=0.16', 'accelerate>=1.6', 'bitsandbytes>=0.45', 'sentencepiece']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if IN_COLAB and not token:
    from google.colab import userdata
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            token = None
        if token:
            break
elif not IN_COLAB:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")

# `HF_TOKEN` is the canonical huggingface_hub variable. The course also sets its
# descriptive alias because some lesson code uses HUGGINGFACE_TOKEN explicitly.
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGINGFACE_TOKEN"] = token

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if True and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Colab runtime detected:", IN_COLAB)
print("Hugging Face token configured:", bool(os.getenv("HF_TOKEN")))
if IN_COLAB and not token:
    print("Add an HF_TOKEN secret in Colab, enable notebook access, then rerun this cell.")


## 21.1 From demonstrations to preferences

SFT learns from desired responses. Preference optimization learns that response
\(y_w\) is preferred to \(y_l\) for prompt \(x\). Preferences can encode correctness,
usefulness, style, or safety, but inconsistent raters and length/style shortcuts become
training signal. Keep ties/ambiguity, rater metadata, and clear guidelines.


## 21.2 DPO intuition

DPO increases the policy's log-probability advantage for chosen over rejected answers,
relative to the same advantage under a reference policy. A common loss is

\[
-\log\sigma\left(\beta[(\log\pi_\theta(y_w|x)-\log\pi_\theta(y_l|x))-
(\log\pi_{ref}(y_w|x)-\log\pi_{ref}(y_l|x))]\right).
\]

\(\beta\) controls strength relative to the reference. DPO avoids an explicit reward
model and online RL loop, but still needs SFT-quality initialization and careful evals.


In [ ]:
import torch
def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=0.1):
    policy_ratio = policy_chosen - policy_rejected
    ref_ratio = ref_chosen - ref_rejected
    logits = beta * (policy_ratio - ref_ratio)
    return -torch.nn.functional.logsigmoid(logits).mean()

loss = dpo_loss(torch.tensor([-2.0]), torch.tensor([-3.0]),
                torch.tensor([-2.5]), torch.tensor([-2.7]))
print(loss.item())


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from datasets import load_dataset
    from peft import LoraConfig
    from trl import DPOConfig, DPOTrainer

    data = load_dataset("trl-lib/ultrafeedback_binarized", split="train[:1000]")
    args = DPOConfig(output_dir="artifacts/qwen-dpo", max_length=1024,
                     per_device_train_batch_size=1, gradient_accumulation_steps=8,
                     learning_rate=5e-6, num_train_epochs=1, fp16=True,
                     gradient_checkpointing=True,
                     eval_strategy="steps", eval_steps=50, report_to="none")
    trainer = DPOTrainer(model="Qwen/Qwen2.5-0.5B-Instruct", args=args,
                         train_dataset=data,
                         peft_config=LoraConfig(task_type="CAUSAL_LM", r=16))
    trainer.train()
else:
    print("DPO skipped; inspect the preference schema and evaluation plan first.")


## 21.3 Preference-data construction

A preference record needs the same prompt/context with chosen and rejected responses. If
prompts differ, the comparison is confounded. Rejections should be plausible alternatives;
trivial bad answers teach superficial separation. Capture criterion, rater, confidence/tie,
response order, model sources, and timestamps. Randomize display order and measure agreement.
Resolve whether “preferred” means more correct, safer, more concise, or simply stylistically
liked—these objectives can conflict.

Audit shortcuts before training: chosen length, headings, disclaimers, refusal phrases,
citations, model-specific style, and lexical markers. A policy can optimize these without
improving substance. Split by prompt/source, deduplicate responses, and keep a human-reviewed
test set. Synthetic AI preferences scale cheaply but inherit judge biases; mix them with human
checks and task-verifiable signals where possible.


In [ ]:
# Preference shortcut audit on a toy set.
pairs = [
    {"chosen": "The answer is 4.", "rejected": "I think it might perhaps be five."},
    {"chosen": "Insufficient evidence.", "rejected": "Definitely Paris, with complete certainty."},
    {"chosen": "Use two microbatches.", "rejected": "Use one batch."},
]
for i, pair in enumerate(pairs):
    c, r = pair["chosen"], pair["rejected"]
    print(i, {"chosen_chars": len(c), "rejected_chars": len(r),
              "chosen_words": len(c.split()), "rejected_words": len(r.split())})


## 21.4 Understanding beta, margins, and reference behavior

DPO compares the policy's chosen/rejected log-probability margin with the reference margin.
If the policy already favors the chosen response more strongly than the reference, its DPO
logit is positive and loss falls. Beta controls sensitivity/regularization conventionally:
larger beta makes a given margin difference produce a more saturated classification signal.
Monitor chosen/rejected rewards, margins, accuracy, log probabilities, and KL-like drift—not
only scalar loss.

The reference policy anchors behavior. With PEFT, implementations may use the base model or
disabled adapter as reference to avoid a full duplicate, but verify library semantics and
memory. DPO assumes preference data follows a Bradley–Terry-style logistic model and does not
explicitly optimize absolute answer quality. Both candidates can be poor; adding SFT/NLL
components or quality filtering can help. Preference optimization can also reduce diversity
or exploit length, so downstream evals remain decisive.


In [ ]:
# Visualize DPO loss across policy-minus-reference preference margins.
import torch, matplotlib.pyplot as plt
margins = torch.linspace(-5, 5, 201)
for beta in [0.05, 0.1, 0.5, 1.0]:
    losses = -torch.nn.functional.logsigmoid(beta * margins)
    plt.plot(margins, losses, label=f"beta={beta}")
plt.xlabel("policy preference margin - reference margin")
plt.ylabel("DPO pair loss"); plt.legend(); plt.show()


## 21.5 Preference optimization family and evaluation

PPO/RLHF trains a reward model and optimizes policy actions with an online RL algorithm;
it is flexible but operationally complex. DPO gives a direct offline objective. IPO changes
the loss to address overfitting behavior. KTO can learn from desirable/undesirable examples
without pairs. ORPO/SimPO and other variants change reference or SFT coupling. GRPO-style
methods compare groups and are used with verifiable/reward signals. Names evolve quickly;
compare assumptions, data requirements, stability, compute, and empirical evals.

Evaluate pairwise win rate with randomized order, objective task metrics, calibration of
refusals, verbosity/length, safety slices, general capability retention, and generation
diversity. Use multiple decoding settings because preference training changes distribution
shape. Compare SFT checkpoint, preference checkpoint, and base. Inspect regressions rather
than reporting a single judge win rate. Roll out gradually: preference optimization can
strongly change tone and refusal behavior even when average benchmarks improve.


## 21.6 Preference-optimization reference

| Method | Data/signal | Distinguishing feature |
|---|---|---|
| Reward modeling + PPO | Preferences then online reward | Explicit reward model and RL loop |
| DPO | Chosen/rejected pairs | Offline policy/reference classification objective |
| IPO/variants | Preference pairs | Alternative regularization/loss assumptions |
| KTO | Desirable/undesirable examples | Does not require paired responses |
| GRPO-style | Group rewards/verifiers | Relative group updates, common in reasoning work |

Names and implementations evolve; use current library documentation and inspect the exact loss/metrics.
Preference accuracy can improve while absolute response quality remains poor. Keep SFT-quality data or
loss, filter both-bad pairs, and evaluate objective tasks. Beta, LR, reference handling, response length,
adapter configuration, and truncation are coupled choices.

Monitor chosen/rejected log probabilities and rewards, margin/accuracy, KL/drift proxies, length,
validation loss, task metrics, safety/refusal behavior, and diversity. Audit order and style shortcuts.
Preference data encodes values and rater context; document who rated what under which guidelines.


## 21.7 DPO margins and reference drift

DPO compares policy log-probability margins between chosen and rejected completions relative to a frozen reference. Inspect chosen/rejected lengths, policy and reference log-probabilities, reward margins, accuracy, and saturation. Large beta emphasizes preference separation but can move farther from the reference; tiny beta may underfit. Sequence log-probability sums introduce length effects, so dataset construction and normalization choices matter. Confirm the reference is the intended SFT checkpoint and never updated by the optimizer.


In [ ]:
policy_c,policy_r,ref_c,ref_r=-2.0,-3.1,-2.4,-2.8
beta=.1; margin=(policy_c-policy_r)-(ref_c-ref_r); loss=-torch.nn.functional.logsigmoid(torch.tensor(beta*margin)); print("margin",margin,"DPO loss",loss.item())


## 21.8 Preference-data diagnostics

Preferences may reflect correctness, style, length, harmlessness, or annotator identity. Keep the prompt identical within a pair, remove duplicates and leakage, quantify position and length bias, and inspect disagreement. Ties or weak preferences should not be forced into confident binary labels. Evaluate reward accuracy by slice and use held-out human or verifiable outcomes. A model can optimize superficial preference cues while task success falls, so retain capability, calibration, and safety suites alongside pairwise win rate.


In [ ]:
pairs=[{"chosen_len":40,"rejected_len":90,"label":1},{"chosen_len":80,"rejected_len":35,"label":1},{"chosen_len":22,"rejected_len":70,"label":1}]
print("shorter chosen rate",sum(x["chosen_len"]<x["rejected_len"] for x in pairs)/len(pairs))


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [Direct Preference Optimization](https://arxiv.org/abs/2305.18290)
- [TRL DPOTrainer](https://huggingface.co/docs/trl/dpo_trainer)


## Exercises

    1. Plot DPO loss as the policy preference margin varies.
2. Audit 100 preference pairs for length, tone, and correctness shortcuts.
3. Compare SFT and DPO checkpoints on both target behavior and capability retention.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
